# 06 — The 1-D Closed Form

On the line, the whole linear program collapses to *sort and match by quantile* — and $W_1$ becomes the plain area between two cumulative curves.

**Prerequisites:** `05_wasserstein_distance`.

**What you'll be able to do**
- Compute $W_p$ in one dimension by the quantile closed form, and confirm it matches the LP exactly.
- Explain the $O((n+m)\log(n+m))$ cost of sort-and-match versus $O(n^3)$ for the LP.
- Read $W_1$ as the area between the two CDFs.

Solving the full linear program for one distance is a lot of machinery. On the line, almost magically, it all collapses: sort both distributions, match them quantile to quantile, and read off the answer. This shortcut — together with the picture of $W_1$ as the area between two cumulative curves — is one of the most useful facts in all of optimal transport.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.transport.discrete import squared_euclidean_cost
from qot_course.transport.wasserstein import wasserstein_distance, wasserstein_1d

np.random.seed(42)
viz.use_course_style()

## 1. Sort and match by quantile

On the line the LP has a closed-form answer. Let $F_\mu, F_\nu$ be the cumulative distribution functions of $\mu, \nu$. The optimal transport map is the **quantile composition** (Brenier; Villani 2003, Thm. 2.18)

$$ T = F_\nu^{-1} \circ F_\mu, $$

which sends each unit of $\mu$-mass at quantile $u$ to the $\nu$-position at the same quantile $u$. The cost integrates to

$$ W_p^p(\mu, \nu) = \int_0^1 \bigl| F_\mu^{-1}(u) - F_\nu^{-1}(u) \bigr|^p\, \mathrm{d}u. $$

For atomic distributions this is a finite sum over the *common refinement* of the two CDFs — computable in $O((n+m)\log(n+m))$ instead of the $O(n^3)$ network simplex. Let's confirm it matches the LP to machine precision.

In [ ]:
rng = np.random.default_rng(11)
n, m = 6, 8
pa = np.sort(rng.uniform(-3.0, 3.0, size=n))   # source atom positions
pb = np.sort(rng.uniform(-3.0, 3.0, size=m))   # target atom positions
wa = rng.random(n); wa /= wa.sum()             # source weights
wb = rng.random(m); wb /= wb.sum()             # target weights

C = squared_euclidean_cost(pa, pb)
w2_lp       = wasserstein_distance(wa, wb, C, p=2)   # full Kantorovich LP
w2_quantile = wasserstein_1d(pa, wa, pb, wb, p=2)    # 1-D quantile closed form

print(f"W2 via Kantorovich LP (POT network simplex) = {w2_lp:.8f}")
print(f"W2 via 1-D quantile closed form             = {w2_quantile:.8f}")
print(f"agreement?  {bool(np.isclose(w2_lp, w2_quantile))}")

**Read the output.** The two methods agree to machine precision. The closed form needed only a sort and a scan of CDF breakpoints; the LP ran a full network-simplex solve. In one dimension the LP is overkill — which is exactly why higher dimensions, where this gift disappears, push us toward the clever algorithms of the next arc.

## 2. $W_1$ as the area between CDFs

For $p = 1$ the closed form has a striking *horizontal* reading (Vallender, 1974):

$$ W_1(\mu, \nu) = \int_{-\infty}^{+\infty} \bigl| F_\mu(x) - F_\nu(x) \bigr|\, \mathrm{d}x. $$

The $W_1$ distance is the area between the two CDFs. Watch it on two well-separated bumps.

In [ ]:
grid = np.linspace(-1.0, 11.0, 600)
dx = grid[1] - grid[0]

def density(center, sigma):
    """Gaussian bump on the grid, normalised to a probability density."""
    pdf = np.exp(-0.5 * ((grid - center) / sigma) ** 2)
    return pdf / np.trapezoid(pdf, grid)

mass_mu = density(3.0, 0.8) * dx; mass_mu /= mass_mu.sum()
mass_nu = density(7.0, 1.2) * dx; mass_nu /= mass_nu.sum()

cdf_mu = np.cumsum(mass_mu)
cdf_nu = np.cumsum(mass_nu)
w1_via_area        = float(np.sum(np.abs(cdf_mu - cdf_nu)) * dx)   # Riemann sum of int |F_mu - F_nu| dx
w1_via_closed_form = wasserstein_1d(grid, mass_mu, grid, mass_nu, p=1)

print(f"W1 via 'area between CDFs'                    = {w1_via_area:.4f}")
print(f"W1 via 1-D quantile closed form (same thing)  = {w1_via_closed_form:.4f}")
print(f"expected ~ 4.0  (the bump centres are 4 apart)")

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(grid, mass_mu / dx, color=viz.SOURCE_COLOR, lw=2, label=r"$\mu$ at $x=3$")
axes[0].plot(grid, mass_nu / dx, color=viz.TARGET_COLOR, lw=2, label=r"$\nu$ at $x=7$")
axes[0].set_ylabel("density")
axes[0].set_title("Two distributions on a line", pad=10)
axes[0].legend()

axes[1].plot(grid, cdf_mu, color=viz.SOURCE_COLOR, lw=2, label=r"$F_\mu$")
axes[1].plot(grid, cdf_nu, color=viz.TARGET_COLOR, lw=2, label=r"$F_\nu$")
axes[1].fill_between(grid, cdf_mu, cdf_nu, color=viz.FLOW_COLOR, alpha=0.32,
                     label=r"$W_1 = \int |F_\mu - F_\nu|\,\mathrm{d}x$")
axes[1].set_xlabel("position  x")
axes[1].set_ylabel("cumulative probability")
axes[1].set_title(f"W1 is the area between the two CDFs  ~ {w1_via_area:.3f}", pad=10)
axes[1].legend(loc="lower right")
plt.tight_layout()
plt.show()

**Read both panels.** The top panel shows two well-separated bumps; the bottom shows their CDFs, a smooth shifted pair. The green shaded region between the curves *is* the $W_1$ distance, and its area agrees with the closed form at $W_1 \approx 4.0$ — the gap between the bump centres. Two readings, one integral: vertical (quantile differences) and horizontal (CDF differences) compute the same number.

## Your turn

1. **The Vallender identity.** For two atomic distributions on the integers, show that the area form $W_1 = \sum_x |F_\mu(x) - F_\nu(x)|$ equals the quantile form $W_1 = \int_0^1 |F_\mu^{-1} - F_\nu^{-1}|\, \mathrm{d}u$. *Hint: integrate by parts.*
2. **$W_2$ between centred Gaussians.** Take $\mu = \mathcal{N}(0, \sigma_\mu^2)$ and $\nu = \mathcal{N}(0, \sigma_\nu^2)$. Discretise on the grid, compute $W_2$, and compare with the closed form $W_2 = |\sigma_\mu - \sigma_\nu|$ — a one-dimensional taste of the Bures–Wasserstein formula coming in `12_bures_wasserstein`.
3. **Feel the complexity gap.** Time `wasserstein_distance` (LP) against `wasserstein_1d` (quantile) as $n$ grows from $10$ to a few thousand. Does the timing match $O(n^3)$ versus $O(n \log n)$?

## What you built

- You computed $W_p$ on the line by the quantile closed form and confirmed it matches the LP exactly.
- You saw why sort-and-match costs $O((n+m)\log(n+m))$ where the LP costs $O(n^3)$.
- You read $W_1$ as the area between two CDFs — a picture you can draw by hand.

You now have both a fast exact algorithm in one dimension and a geometric way to *see* the distance. Keep the closed form in mind: when the LP is too heavy in higher dimensions, the next arc recovers speed a different way.

## What's next

In `07_mccann_geodesic` we close the Wasserstein arc with its most beautiful feature: the $W_2$ geodesic *slides* mass through the ground space — a single bump moving from source to target — which is the transport-geometry half of the two-geometries hinge from `02/08`.

## References

- C. Villani, *Topics in Optimal Transportation*, AMS (2003), ch. 2 and Thm. 2.18.
- S. S. Vallender, "Calculation of the Wasserstein distance between probability distributions on the line", *Theory Probab. Appl.* 18, 784–786 (1974).
- G. Peyré & M. Cuturi, *Computational Optimal Transport*, NoW (2019), ch. 2. DOI:10.1561/2200000073

**Previous:** `notebooks/03_ClassicalOptimalTransport/05_wasserstein_distance.ipynb` · **Next:** `notebooks/03_ClassicalOptimalTransport/07_mccann_geodesic.ipynb`